# M4 Training from Scratch - 3-Layer Strategy V2

**Target**: Accuracy 75% → >95% | 6→0 Error Rate: 20.4% → <5%

## ⚠️ IMPORTANT: Training from Scratch (NOT Fine-tuning)

This notebook trains a **brand new model from scratch** using the 3-Layer Strategy.
- No pre-trained weights are loaded
- Model learns from random initialization
- Architecture: MobileNet V1 + MultiHead (CTC + SAR)
- Training time: ~30-45 minutes on GPU T4

## 🎯 3-Layer Strategy V2

### Layer 1: WeightedRandomSampler - Balance "Diet" 🍽️
- Force AI to see rare digits (6,7,8,9) equally with common digits (0,1,2)
- Prevents AI from "guessing" 0 for high probability

### Layer 2: Sharpening Pre-processing - "Magnifying Glass" 🔍
- Apply Unsharp Masking + CLAHE to blurry images
- Enhances blurry 6→0 features (the "hook" on top of 6)

### Layer 3: Digit Weighted Loss - Increase "Punishment" ⚖️
- 📊 **Optimized weights based on failure analysis:**
  - Digit 6: **4.0x** (Tỉ lệ lỗi 26% - CAO NHẤT) - 8x vs digit 0
  - Digit 1: **3.5x** (Số lượng lỗi tuyệt đối nhiều nhất: 370 lần)
  - Digit 8: **3.0x** (Hay bị nhầm thành 0 và 1)
  - Digit 9: **2.5x** (Dữ liệu ít, dễ bị bỏ qua)
  - Digits 4,5,7: **2.0x** (Hay nhầm chéo)
  - Digits 2,3: **1.5x** (Tương đối ổn định)
  - Digit 0: **0.5x** (GIẢM TRỌNG SỐ - AI bớt đoán mò)

## 🏗️ New Architecture: MobileNet V1 Enhanced + MultiHead

- **Backbone**: MobileNet V1 Enhanced (lightweight, fast)
- **MultiHead**: CTC + SAR (Show, Attend and Read)
- **Input**: [3, 48, 192] (RGB, wider for better digit separation)
- **Learning Rate**: 1e-3 with CosineAnnealingLR scheduler
- **Training**: From scratch (no pre-trained weights)

---

**Expected Results:**
- 6→0 error rate: 20.4% → <5%
- 1→0 error rate: 19.2% → <5%
- 8→0 error rate: 9.7% → <3%
- Overall OCR accuracy: 76.55% → >90%
- Pipeline success: 96.25% → >98%
- Training time: ~30-45 minutes on GPU T4

## 📦 Setup - Mount Google Drive & Install Dependencies

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

print("✅ Google Drive mounted successfully!")

In [ ]:
# Install dependencies
!pip install torch torchvision torchaudio -q
!pip install opencv-python-headless -q
!pip install tqdm pandas -q

import os
import sys
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import cv2
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm
from collections import Counter
import json

print("✅ All dependencies installed!")
print(f"✅ PyTorch version: {torch.__version__}")
print(f"✅ CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
ZIP_FILE_PATH = '/content/drive/MyDrive/WaterMeter_Project/M4_Training/data/m4_ocr_dataset_black_digits.zip'
EXTRACT_DIR = './data'

print(f"Unzipping {ZIP_FILE_PATH} to {EXTRACT_DIR}...")
!mkdir -p {EXTRACT_DIR}
!unzip -q -o {ZIP_FILE_PATH} -d {EXTRACT_DIR}
print("✅ Dataset unzipped successfully!")

# Define paths
DATA_DIR = f'{EXTRACT_DIR}/m4_ocr_dataset_black_digits/images'
LABELS_FILE = f'{EXTRACT_DIR}/m4_ocr_dataset_black_digits/labels.csv'
OUTPUT_DIR = './output_models'
!mkdir -p {OUTPUT_DIR}

print(f"\nDATA_DIR: {DATA_DIR}")
print(f"LABELS_FILE: {LABELS_FILE}")
print(f"OUTPUT_DIR: {OUTPUT_DIR}")

## 🏗️ MobileNet V1 Enhanced + MultiHead Architecture

**Key Improvements:**
- MobileNet V1 backbone (lightweight, efficient)
- MultiHead: CTC + SAR (Attention)
- Input: [3, 48, 192] (RGB, wider for better context)
- Depthwise separable convolutions for efficiency

In [ ]:
class DepthwiseSeparableConv(nn.Module):
    """Depthwise separable convolution - lighter than standard conv"""
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        self.depthwise = nn.Sequential(
            nn.Conv2d(in_channels, in_channels, kernel_size=3, stride=stride, 
                     padding=1, groups=in_channels, bias=False),
            nn.BatchNorm2d(in_channels),
            nn.ReLU6(inplace=True)
        )
        self.pointwise = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU6(inplace=True)
        )
    
    def forward(self, x):
        x = self.depthwise(x)
        x = self.pointwise(x)
        return x


class MobileNetV1Encoder(nn.Module):
    """
    MobileNet V1 Enhanced Encoder
    Input: [3, 48, 192]
    """
    def __init__(self, input_channels=3):
        super().__init__()
        
        # Initial conv
        self.conv1 = nn.Sequential(
            nn.Conv2d(input_channels, 32, kernel_size=3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU6(inplace=True)
        )
        
        # MobileNet V1 blocks (depthwise separable)
        # Format: (in_channels, out_channels, stride)
        self.dw_conv_layers = nn.ModuleList([
            DepthwiseSeparableConv(32, 64, stride=1),      # [64, 24, 96]
            DepthwiseSeparableConv(64, 128, stride=2),     # [128, 12, 48]
            DepthwiseSeparableConv(128, 128, stride=1),    # [128, 12, 48]
            DepthwiseSeparableConv(128, 256, stride=2),    # [256, 6, 24]
            DepthwiseSeparableConv(256, 256, stride=1),    # [256, 6, 24]
            DepthwiseSeparableConv(256, 512, stride=2),    # [512, 3, 12]
            DepthwiseSeparableConv(512, 512, stride=1),    # [512, 3, 12]
            DepthwiseSeparableConv(512, 512, stride=1),    # [512, 3, 12]
            DepthwiseSeparableConv(512, 512, stride=1),    # [512, 3, 12]
            DepthwiseSeparableConv(512, 512, stride=1),    # [512, 3, 12]
        ])
        
    def forward(self, x):
        # x: [B, 3, 48, 192]
        x = self.conv1(x)  # [B, 32, 24, 96]
        
        intermediate_features = []
        for i, layer in enumerate(self.dw_conv_layers):
            x = layer(x)
            # Save features at certain resolutions for multi-scale
            if i in [2, 4, 6]:  # After strides
                intermediate_features.append(x)
        
        # x: [B, 512, 3, 12]
        return x, intermediate_features


class MultiHeadOCR(nn.Module):
    """
    Multi-Head OCR with CTC + SAR (Show, Attend and Read)
    
    Heads:
    1. CTC Head: For sequence alignment
    2. SAR Head: For attention-based decoding (better for 6→0 fix)
    """
    def __init__(self, num_chars=11, hidden_size=256, num_layers=2):
        super().__init__()
        self.num_chars = num_chars
        self.hidden_size = hidden_size
        
        # Encoder
        self.encoder = MobileNetV1Encoder(input_channels=3)
        
        # Feature projection
        # Input feature: [B, 512, 3, 12] -> [B, 512*3, 12] -> [B, 12, 1536]
        self.feature_proj = nn.Linear(512 * 3, hidden_size)
        
        # Bidirectional LSTM for sequence modeling
        self.rnn = nn.LSTM(
            input_size=hidden_size,
            hidden_size=hidden_size // 2,
            num_layers=num_layers,
            bidirectional=True,
            batch_first=True,
            dropout=0.3
        )
        
        # ==================== CTC HEAD ====================
        self.ctc_head = nn.Linear(hidden_size, num_chars)
        
        # ==================== SAR HEAD ====================
        # Attention mechanism
        self.attention = nn.Sequential(
            nn.Linear(hidden_size, hidden_size),
            nn.Tanh(),
            nn.Linear(hidden_size, 1)
        )
        
        # SAR decoder (autoregressive)
        self.sar_embedding = nn.Embedding(num_chars, hidden_size)
        self.sar_rnn = nn.LSTM(
            input_size=hidden_size * 2,  # embedding + context
            hidden_size=hidden_size,
            num_layers=1,
            batch_first=True
        )
        self.sar_output = nn.Linear(hidden_size, num_chars)
        
        self.max_length = 10  # Max digits to predict
        
    def forward(self, x, sar_targets=None):
        """
        Args:
            x: [B, 3, 48, 192] input images
            sar_targets: [B, max_len] target sequences (for SAR training)
        
        Returns:
            ctc_logits: [T, B, num_chars] for CTC loss
            sar_logits: [B, max_len, num_chars] for SAR loss
        """
        batch_size = x.size(0)
        
        # Encode
        features, _ = self.encoder(x)  # [B, 512, 3, 12]
        
        # Reshape features: [B, 512, 3, 12] -> [B, 12, 1536]
        b, c, h, w = features.size()
        features = features.view(b, c * h, w)  # [B, 1536, 12]
        features = features.permute(0, 2, 1)  # [B, 12, 1536]
        
        # Project and RNN
        features = self.feature_proj(features)  # [B, 12, hidden_size]
        rnn_out, _ = self.rnn(features)  # [B, 12, hidden_size]
        
        # ==================== CTC HEAD ====================
        ctc_logits = self.ctc_head(rnn_out)  # [B, 12, num_chars]
        ctc_logits = ctc_logits.permute(1, 0, 2)  # [T, B, C] for CTC loss
        
        # ==================== SAR HEAD ====================
        if self.training:
            # Training mode: Teacher forcing
            sar_logits = self._train_sar(rnn_out, sar_targets)
        else:
            # Inference mode: Autoregressive decoding
            sar_logits = self._infer_sar(rnn_out)
        
        return ctc_logits, sar_logits
    
    def _train_sar(self, encoder_out, targets):
        """Training mode with teacher forcing"""
        batch_size = encoder_out.size(0)
        
        # Start with SOS token (last token = num_chars - 1 as blank)
        # Use blank token as start
        sos_token = torch.zeros(batch_size, 1, dtype=torch.long, device=encoder_out.device)
        
        # Embed targets
        if targets is not None:
            target_embed = self.sar_embedding(targets)  # [B, max_len, hidden]
        else:
            # Fallback: create dummy targets
            target_embed = torch.zeros(batch_size, self.max_length, self.hidden_size, 
                                       device=encoder_out.device)
        
        # Concatenate SOS
        target_embed = torch.cat([self.sar_embedding(sos_token.squeeze(1).unsqueeze(1)), 
                                 target_embed], dim=1)  # [B, max_len+1, hidden]
        
        # Initialize hidden state
        hidden = None
        
        outputs = []
        context = encoder_out.mean(dim=1, keepdim=True).expand(-1, encoder_out.size(1), -1)
        
        for t in range(self.max_length):
            # Get input embedding
            input_t = target_embed[:, t, :].unsqueeze(1)  # [B, 1, hidden]
            
            # Concatenate with context
            input_ctx = torch.cat([input_t, context[:, 0:1, :]], dim=2)  # [B, 1, 2*hidden]
            
            # RNN step
            output, hidden = self.sar_rnn(input_ctx, hidden)
            
            # Output prediction
            out = self.sar_output(output)  # [B, 1, num_chars]
            outputs.append(out)
        
        sar_logits = torch.cat(outputs, dim=1)  # [B, max_len, num_chars]
        return sar_logits
    
    def _infer_sar(self, encoder_out):
        """Inference mode with autoregressive decoding"""
        batch_size = encoder_out.size(0)
        device = encoder_out.device
        
        # Start with blank token
        current_token = torch.zeros(batch_size, dtype=torch.long, device=device)
        
        # Hidden state
        hidden = None
        
        outputs = []
        
        for t in range(self.max_length):
            # Embed current token
            token_embed = self.sar_embedding(current_token).unsqueeze(1)  # [B, 1, hidden]
            
            # Attention over encoder outputs
            # Compute attention weights
            att_scores = self.attention(encoder_out).squeeze(-1)  # [B, seq_len]
            att_weights = torch.softmax(att_scores, dim=1)  # [B, seq_len]
            
            # Compute context as weighted sum
            context = torch.bmm(att_weights.unsqueeze(1), encoder_out)  # [B, 1, hidden]
            
            # Concatenate token embedding with context
            rnn_input = torch.cat([token_embed, context], dim=2)  # [B, 1, 2*hidden]
            
            # RNN step
            output, hidden = self.sar_rnn(rnn_input, hidden)
            
            # Predict next token
            logits = self.sar_output(output)  # [B, 1, num_chars]
            outputs.append(logits)
            
            # Get predicted token (greedy)
            current_token = logits.argmax(dim=2).squeeze(1)  # [B]
        
        sar_logits = torch.cat(outputs, dim=1)  # [B, max_len, num_chars]
        return sar_logits


# Test model
model = MultiHeadOCR(num_chars=11, hidden_size=256)
x = torch.randn(2, 3, 48, 192)
ctc_out, sar_out = model(x)
print(f"Input shape: {x.shape}")
print(f"CTC output shape: {ctc_out.shape}")  # [T, B, C]
print(f"SAR output shape: {sar_out.shape}")  # [B, max_len, C]
print("\n✅ MultiHead OCR model initialized!")

## 🎯 Layer 3: Weighted Loss - Higher Punishment for Digit 6 ⚖️

In [ ]:
class DigitWeightedCTCLoss(nn.Module):
    """
    Layer 3: Weighted CTC Loss - Higher punishment for digit 6 errors

    📊 BẢNG TRỌNG SỐ ĐỀ XUẤT (Dựa trên phân tích lỗi thực tế)

    Chữ số | Trọng số | Lý do
    --------|----------|-------
      6    |   4.0    | Tỉ lệ lỗi 26% (CAO NHẤT) - Target chính!
      1    |   3.5    | Số lượng lỗi tuyệt đối nhiều nhất (370 lần)
      8    |   3.0    | Hay bị nhầm thành 0 và 1
      9    |   2.5    | Dữ liệu ít, dễ bị mô hình bỏ qua
      7    |   2.0    | Hay nhầm với 1
      5    |   2.0    | Hay nhầm thành 8 (mirror)
      4    |   2.0    | Hay nhầm thành 3
      2    |   1.5    | Tương đối ổn định
      3    |   1.5    | Tương đối ổn định
      0    |   0.5    | GIẢM TRỌNG SỐ - AI bớt "đoán mò" là 0

    Note: Digit 6 có trọng số 8x cao hơn digit 0!
    """
    def __init__(self, digit_weights=None, blank_idx=10):
        super().__init__()
        if digit_weights is None:
            # 📊 TRỌNG SỐ ĐỀ XUẤT - Dựa trên phân tích lỗi thực tế
            self.digit_weights = torch.tensor([
                0.5,  # 0: GIẢM TRỌNG SỐ - quá phổ biến, AI hay đoán mò
                3.5,  # 1: Số lượng lỗi tuyệt đối nhiều nhất (370 lần)
                1.5,  # 2: Tương đối ổn định
                1.5,  # 3: Tương đối ổn định
                2.0,  # 4: Hay nhầm thành 3
                2.0,  # 5: Hay nhầm thành 8
                4.0,  # 6: TARGET! Tỉ lệ lỗi 26% (cao nhất) - 8x vs 0
                2.0,  # 7: Hay nhầm thành 1
                3.0,  # 8: Hay bị nhầm thành 0 và 1
                2.5,  # 9: Dữ liệu ít, dễ bị bỏ qua
                1.0,  # 10: Blank (CTC)
            ])
        else:
            self.digit_weights = torch.tensor(digit_weights)

        self.blank_idx = blank_idx
        self.ctc_loss = nn.CTCLoss(blank=blank_idx, reduction='none')

    def forward(self, logits, targets, input_lengths, target_lengths):
        """
        Args:
            logits: (T, N, C) - T time steps, N batch, C classes
            targets: Concatenated target labels
            input_lengths: Length of each sequence
            target_lengths: Length of each target

        Returns:
            Weighted loss
        """
        log_probs = logits.log_softmax(2)

        # Calculate base CTC loss (per sample)
        losses = []
        for i in range(logits.size(1)):
            target_i = targets[
                sum(target_lengths[:i]):sum(target_lengths[:i+1])
            ]
            loss_i = self.ctc_loss(
                log_probs[:, i:i+1, :],
                target_i.unsqueeze(0),
                input_lengths[i:i+1],
                target_lengths[i:i+1]
            )

            # Apply weight based on target digits
            weights = self.digit_weights.to(logits.device)[target_i]
            avg_weight = weights.mean().to(logits.device)

            # Weighted loss: higher weight for rare digits
            weighted_loss = loss_i * avg_weight
            losses.append(weighted_loss)

        return torch.stack(losses).mean()


class MultiHeadLoss(nn.Module):
    """
    Combined loss for MultiHead OCR
    - CTC Loss (weighted) for alignment
    - CrossEntropy Loss for SAR head
    """
    def __init__(self, ctc_weight=0.5, sar_weight=0.5):
        super().__init__()
        self.ctc_weight = ctc_weight
        self.sar_weight = sar_weight
        self.ctc_criterion = DigitWeightedCTCLoss(blank_idx=10)
        self.sar_criterion = nn.CrossEntropyLoss(ignore_index=-1)
    
    def forward(self, ctc_logits, sar_logits, targets, input_lengths, target_lengths):
        # CTC loss
        ctc_loss = self.ctc_criterion(ctc_logits, targets, input_lengths, target_lengths)
        
        # SAR loss (need to pad targets to max_length)
        batch_size = sar_logits.size(0)
        max_len = sar_logits.size(1)
        
        # Prepare SAR targets
        sar_targets = torch.full((batch_size, max_len), -1, dtype=torch.long, device=sar_logits.device)
        for i in range(batch_size):
            target_start = sum(target_lengths[:i])
            target_end = target_start + target_lengths[i]
            target_i = targets[target_start:target_end]
            
            # Pad or truncate to max_len
            if target_i.size(0) <= max_len:
                sar_targets[i, :target_i.size(0)] = target_i
            else:
                sar_targets[i, :] = target_i[:max_len]
        
        # Reshape SAR logits for CrossEntropyLoss
        sar_logits_flat = sar_logits.reshape(-1, sar_logits.size(2))
        sar_targets_flat = sar_targets.reshape(-1)
        
        sar_loss = self.sar_criterion(sar_logits_flat, sar_targets_flat)
        
        # Combined loss
        total_loss = self.ctc_weight * ctc_loss + self.sar_weight * sar_loss
        
        return total_loss, ctc_loss, sar_loss

# Initialize loss
criterion = MultiHeadLoss(ctc_weight=0.5, sar_weight=0.5)
print("✅ MultiHead Loss initialized!")
print("   📊 Digit weights (based on failure analysis):")
for i, w in enumerate(criterion.ctc_criterion.digit_weights):
    if i < 10:
        print(f"     Digit {i}: {w.item():.1f}x", end="")
        if i == 6:
            print(" ← TARGET! (8x vs 0)")
        elif i == 0:
            print(" (reduced - AI stops guessing)")
        else:
            print()
print()

## 🔍 Layer 2: Sharpening Pre-processing - "Magnifying Glass" 🔍

In [ ]:
class SharpenTransform:
    """
    Layer 2: Sharpening Pre-processing - "Magnifying Glass" for blurry 6→0

    Techniques:
    1. Unsharp Masking: Enhance edges and fine details
    2. CLAHE: Improve local contrast
    3. Denoising: Remove noise while preserving edges

    Target: Make the "hook" on top of 6 more visible
    """
    def __init__(self, sharpen_strength=1.5, clahe_clip=2.0):
        self.sharpen_strength = sharpen_strength
        self.clahe_clip = clahe_clip

    def __call__(self, image: np.ndarray) -> np.ndarray:
        # Convert to grayscale if needed
        if len(image.shape) == 3:
            gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
        else:
            gray = image.copy()

        # Step 1: Denoising (light)
        denoised = cv2.fastNlMeansDenoising(gray, None, h=5, templateWindowSize=7, searchWindowSize=21)

        # Step 2: Unsharp Masking (Sharpening)
        # Create a blurred version
        blurred = cv2.GaussianBlur(denoised, (0, 0), 3)
        # Sharpen: original + strength * (original - blurred)
        sharpened = cv2.addWeighted(
            denoised,
            1 + self.sharpen_strength,
            blurred,
            -self.sharpen_strength,
            0
        )

        # Step 3: CLAHE (Contrast Limited Adaptive Histogram Equalization)
        clahe = cv2.createCLAHE(clipLimit=self.clahe_clip, tileGridSize=(4, 4))
        enhanced = clahe.apply(sharpened)

        return enhanced

# Initialize sharpening transform
sharpen_transform = SharpenTransform(sharpen_strength=1.5, clahe_clip=2.0)
print("✅ Layer 2: Sharpening transform initialized!")
print(f"   Sharpen strength: 1.5")
print(f"   CLAHE clip limit: 2.0")
print(f"   Denoising: Enabled")

## 📊 Layer 1: Dataset with WeightedRandomSampler - Balance "Diet" 🍽️

In [ ]:
class MeterDataset(Dataset):
    """
    Dataset with digit-based sampling strategy

    Layer 1: Samples with rare digits (6,7,8,9) get higher weights
    This forces AI to see rare digits equally with common digits (0,1,2)
    
    Enhanced for RGB input [3, 48, 192]
    """
    def __init__(self, images_dir, labels_file, transform=None, sharpen=None):
        self.images_dir = Path(images_dir)
        self.transform = transform
        self.sharpen = sharpen

        # Load labels
        self.labels_df = pd.read_csv(labels_file)
        self.samples = []

        # Calculate digit frequencies
        digit_counts = Counter()
        for _, row in self.labels_df.iterrows():
            value = str(row['text'])
            for digit in value:
                if digit.isdigit():
                    digit_counts[int(digit)] += 1

        print("Digit distribution in dataset:")
        total_digits = sum(digit_counts.values())
        for digit in sorted(digit_counts.keys()):
            count = digit_counts[digit]
            print(f"  Digit {digit}: {count:5d} occurrences ({count/total_digits*100:5.2f}%)")

        # Calculate sample weights (inverse of digit frequency)
        # Samples with rare digits get higher weights
        self.sample_weights = []

        for _, row in self.labels_df.iterrows():
            value = str(row['text'])

            # Calculate weight based on rarest digit in this sample
            min_freq = float('inf')
            for digit in value:
                if digit.isdigit():
                    freq = digit_counts[int(digit)]
                    min_freq = min(min_freq, freq)

            # Inverse frequency: rarer digits = higher weight
            weight = 1.0 / (min_freq + 1e-6)
            self.sample_weights.append(weight)

        # Normalize weights
        total_weight = sum(self.sample_weights)
        self.sample_weights = [w / total_weight for w in self.sample_weights]

        print(f"\nSample weight range: {min(self.sample_weights):.6f} - {max(self.sample_weights):.6f}")

    def __len__(self):
        return len(self.labels_df)

    def __getitem__(self, idx):
        row = self.labels_df.iloc[idx]
        img_path = self.images_dir / row['filename']

        # Load image
        image = cv2.imread(str(img_path))
        if image is None:
            raise ValueError(f"Cannot load image: {img_path}")

        # Apply sharpening (Layer 2)
        if self.sharpen is not None:
            gray_sharpened = self.sharpen(image)
            # Convert back to 3-channel RGB
            image_rgb = cv2.cvtColor(gray_sharpened, cv2.COLOR_GRAY2RGB)
        else:
            # Convert to RGB
            image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        # Resize to [192, 48] (width, height)
        image_rgb = cv2.resize(image_rgb, (192, 48))

        # Normalize to [-1, 1]
        image_rgb = image_rgb.astype(np.float32) / 255.0
        image_rgb = (image_rgb - 0.5) / 0.5

        # Get label
        label = str(row['text'])

        # Convert label to indices
        label_indices = [int(d) for d in label if d.isdigit()]

        return {
            'image': torch.from_numpy(image_rgb).permute(2, 0, 1).float(),  # [3, 48, 192]
            'label': torch.tensor(label_indices, dtype=torch.long),
            'label_length': len(label_indices),
            'text': label
        }

## 🔄 Helper Functions

In [ ]:
def collate_fn(batch):
    """Custom collate function for variable-length sequences"""
    images = torch.stack([item['image'] for item in batch])
    labels = torch.cat([item['label'] for item in batch])
    label_lengths = torch.tensor([item['label_length'] for item in batch])
    input_lengths = torch.tensor([12] * len(batch), dtype=torch.long)  # Sequence length after encoder

    return {
        'images': images,
        'labels': labels,
        'input_lengths': input_lengths,
        'label_lengths': label_lengths,
        'texts': [item['text'] for item in batch]
    }


def decode_ctc(pred_logits, idx_to_char=None):
    """Decode CTC predictions (greedy)"""
    preds = pred_logits.argmax(dim=2)  # [T, B]
    
    results = []
    for b in range(preds.size(1)):
        pred_seq = preds[:, b]
        
        # Remove blanks (10) and consecutive duplicates
        text = []
        last_char = None
        for p in pred_seq:
            p = p.item()
            if p != 10 and p != last_char:  # 10 is blank
                text.append(str(p) if idx_to_char is None else idx_to_char.get(p, ''))
                last_char = p
        
        results.append(''.join(text))
    
    return results


def decode_sar(pred_logits):
    """Decode SAR predictions (greedy)"""
    preds = pred_logits.argmax(dim=2)  # [B, max_len, num_chars] -> [B, max_len]
    
    results = []
    for b in range(preds.size(0)):
        text = []
        for p in preds[b]:
            p = p.item()
            if p < 10:  # Not blank
                text.append(str(p))
            else:
                break  # Stop at EOS/blank
        results.append(''.join(text))
    
    return results


def ensemble_decode(ctc_logits, sar_logits, ctc_weight=0.5):
    """
    Ensemble decoding: combine CTC and SAR predictions
    SAR is better for fixing 6→0 errors (attention-based)
    """
    ctc_texts = decode_ctc(ctc_logits)
    sar_texts = decode_sar(sar_logits)
    
    results = []
    for ctc_text, sar_text in zip(ctc_texts, sar_texts):
        # Simple ensemble: prefer SAR if it contains '6' (our target)
        if '6' in sar_text and len(sar_text) > 0:
            results.append(sar_text)
        elif len(ctc_text) >= len(sar_text) and len(ctc_text) > 0:
            results.append(ctc_text)
        elif len(sar_text) > 0:
            results.append(sar_text)
        else:
            results.append(ctc_text if len(ctc_text) > 0 else '0')
    
    return results


print("✅ Helper functions defined!")

## 🚀 Load Model & Data

In [ ]:
# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print()

# ========================================
# ⚠️ TRAINING FROM SCRATCH - NO PRE-TRAINED WEIGHTS
# ========================================
print("="*70)
print("⚠️  TRAINING FROM SCRATCH - NO PRE-TRAINED WEIGHTS")
print("="*70)
print("\nInitializing new model from random initialization...")
print("This will take ~30-45 minutes on GPU T4")
print()

# Create NEW model from scratch (NO pre-trained weights)
model = MultiHeadOCR(num_chars=11, hidden_size=256).to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"✅ New model initialized from scratch!")
print(f"   Total parameters: {total_params:,}")
print(f"   Trainable parameters: {trainable_params:,}")
print()

# Load dataset
print("[Loading Dataset with Layer 1: WeightedRandomSampler]")
dataset = MeterDataset(
    images_dir=DATA_DIR,
    labels_file=LABELS_FILE,
    sharpen=sharpen_transform  # Layer 2: Sharpening
)
print()

# Split train/val
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = torch.utils.data.random_split(dataset, [train_size, val_size])

# Layer 1: Create WeightedRandomSampler for training
train_sampler = WeightedRandomSampler(
    weights=[dataset.sample_weights[i] for i in train_dataset.indices],
    num_samples=len(train_dataset),
    replacement=True
)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    sampler=train_sampler,
    collate_fn=collate_fn,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=2,
    pin_memory=True
)

print(f"✅ Train samples: {len(train_dataset)}")
print(f"✅ Val samples: {len(val_dataset)}")
print(f"✅ Layer 1: WeightedRandomSampler applied - rare digits will be seen more often!")
print()
print("="*70)
print("READY TO TRAIN FROM SCRATCH!")
print("="*70)

## 🏋️ Training & Validation Functions

In [ ]:
def train_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    total_ctc_loss = 0
    total_sar_loss = 0
    total_correct = 0
    total_samples = 0

    # Digit-wise tracking
    digit_correct = 0
    digit_total = 0

    pbar = tqdm(dataloader, desc="Training")
    for batch in pbar:
        images = batch['images'].to(device)
        labels = batch['labels'].to(device)
        input_lengths = batch['input_lengths'].to(device)
        label_lengths = batch['label_lengths'].to(device)

        # Forward pass
        ctc_logits, sar_logits = model(images)

        # Calculate loss
        loss, ctc_loss, sar_loss = criterion(ctc_logits, sar_logits, labels, input_lengths, label_lengths)

        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()

        total_loss += loss.item()
        total_ctc_loss += ctc_loss.item()
        total_sar_loss += sar_loss.item()

        # Calculate accuracy (ensemble decoding)
        pred_texts = ensemble_decode(ctc_logits, sar_logits)
        for pred_text, true_text in zip(pred_texts, batch['texts']):
            if pred_text == true_text:
                total_correct += 1
            total_samples += 1

            # Digit-wise accuracy
            for t, p in zip(true_text, pred_text):
                if t == p:
                    digit_correct += 1
                digit_total += 1

        pbar.set_postfix({
            'loss': f'{loss.item():.4f}',
            'acc': f'{total_correct/total_samples:.4f}'
        })

    return {
        'loss': total_loss / len(dataloader),
        'ctc_loss': total_ctc_loss / len(dataloader),
        'sar_loss': total_sar_loss / len(dataloader),
        'accuracy': total_correct / total_samples,
        'digit_accuracy': digit_correct / digit_total if digit_total > 0 else 0
    }


def validate(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0
    total_ctc_loss = 0
    total_sar_loss = 0
    total_correct = 0
    total_samples = 0

    # Digit-wise accuracy
    digit_correct = 0
    digit_total = 0

    # Track errors
    six_to_zero_errors = 0
    total_six = 0
    one_to_zero_errors = 0
    total_one = 0
    eight_to_zero_errors = 0
    total_eight = 0

    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Validating"):
            images = batch['images'].to(device)
            labels = batch['labels'].to(device)
            input_lengths = batch['input_lengths'].to(device)
            label_lengths = batch['label_lengths'].to(device)

            # Forward pass
            ctc_logits, sar_logits = model(images)

            # Calculate loss
            loss, ctc_loss, sar_loss = criterion(ctc_logits, sar_logits, labels, input_lengths, label_lengths)
            total_loss += loss.item()
            total_ctc_loss += ctc_loss.item()
            total_sar_loss += sar_loss.item()

            # Calculate accuracy
            pred_texts = ensemble_decode(ctc_logits, sar_logits)
            for pred_text, true_text in zip(pred_texts, batch['texts']):
                if pred_text == true_text:
                    total_correct += 1
                total_samples += 1

                # Digit-wise accuracy & error tracking
                for t, p in zip(true_text, pred_text):
                    if t == '6':
                        total_six += 1
                    elif t == '1':
                        total_one += 1
                    elif t == '8':
                        total_eight += 1

                    if t == p:
                        digit_correct += 1
                    digit_total += 1

                    # Track specific errors
                    if t == '6' and p == '0':
                        six_to_zero_errors += 1
                    elif t == '1' and p == '0':
                        one_to_zero_errors += 1
                    elif t == '8' and p == '0':
                        eight_to_zero_errors += 1

    metrics = {
        'loss': total_loss / len(dataloader),
        'ctc_loss': total_ctc_loss / len(dataloader),
        'sar_loss': total_sar_loss / len(dataloader),
        'accuracy': total_correct / total_samples,
        'digit_accuracy': digit_correct / digit_total if digit_total > 0 else 0,
        'six_to_zero_error_rate': six_to_zero_errors / total_six if total_six > 0 else 0,
        'one_to_zero_error_rate': one_to_zero_errors / total_one if total_one > 0 else 0,
        'eight_to_zero_error_rate': eight_to_zero_errors / total_eight if total_eight > 0 else 0
    }

    return metrics


print("✅ Training & validation functions defined!")

## 🎯 Start Training from Scratch with 3-Layer Strategy!

In [ ]:
# Configuration
NUM_EPOCHS = 15
LEARNING_RATE = 1e-3  # Higher for training from scratch
BATCH_SIZE = 32

# Optimizer
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-5)

# Learning rate scheduler (Cosine Annealing with Warm Restarts)
scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer,
    T_0=5,  # Initial restart period
    T_mult=2,  # Double period after each restart
    eta_min=1e-6  # Minimum learning rate
)

print(f"Optimizer: Adam (lr={LEARNING_RATE})")
print(f"Scheduler: CosineAnnealingWarmRestarts (T_0=5, T_mult=2, eta_min=1e-6)")
print(f"  Higher LR - training from scratch with new architecture")
print()

print("="*70)
print("TRAINING FROM SCRATCH WITH 3-LAYER STRATEGY V2")
print("="*70)
print("\n🏗️ Architecture: MobileNet V1 + MultiHead (CTC + SAR)")
print("📏 Input: [3, 48, 192] (RGB, wider)")
print("\nLayer 1: WeightedRandomSampler - Force AI to see rare digits equally")
print("Layer 2: Sharpening + Denoising - Enhance blurry 6→0 features")
print("Layer 3: Weighted Loss - 4x punishment for digit 6 errors")
print("="*70)
print()

best_val_acc = 0
best_six_to_zero_rate = float('inf')
history = {'train_loss': [], 'val_loss': [], 'val_acc': [], 'six_to_zero': []}

for epoch in range(NUM_EPOCHS):
    print(f"\n{'='*70}")
    print(f"EPOCH {epoch+1}/{NUM_EPOCHS}")
    print(f"LR: {scheduler.get_last_lr()[0]:.6f}")
    print(f"{'='*70}")

    # Train
    train_metrics = train_epoch(
        model, train_loader, criterion, optimizer, device
    )

    # Validate
    val_metrics = validate(model, val_loader, criterion, device)

    # Update learning rate
    scheduler.step()

    # Save history
    history['train_loss'].append(train_metrics['loss'])
    history['val_loss'].append(val_metrics['loss'])
    history['val_acc'].append(val_metrics['accuracy'])
    history['six_to_zero'].append(val_metrics['six_to_zero_error_rate'])

    print(f"\n{'='*70}")
    print(f"RESULTS:")
    print(f"{'='*70}")
    print(f"Train Loss: {train_metrics['loss']:.4f} | Train Acc: {train_metrics['accuracy']:.4f}")
    print(f"  CTC: {train_metrics['ctc_loss']:.4f} | SAR: {train_metrics['sar_loss']:.4f}")
    print(f"Val   Loss: {val_metrics['loss']:.4f} | Val   Acc: {val_metrics['accuracy']:.4f}")
    print(f"  CTC: {val_metrics['ctc_loss']:.4f} | SAR: {val_metrics['sar_loss']:.4f}")
    print(f"\n📊 Digit Accuracy: {val_metrics['digit_accuracy']:.2%}")
    print(f"🎯 6→0 Error Rate: {val_metrics['six_to_zero_error_rate']:.2%} ← TARGET!")
    print(f"🎯 1→0 Error Rate: {val_metrics['one_to_zero_error_rate']:.2%}")
    print(f"🎯 8→0 Error Rate: {val_metrics['eight_to_zero_error_rate']:.2%}")

    # Save best model (by accuracy)
    if val_metrics['accuracy'] > best_val_acc:
        best_val_acc = val_metrics['accuracy']
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_acc': val_metrics['accuracy'],
            'val_loss': val_metrics['loss'],
            'six_to_zero_error_rate': val_metrics['six_to_zero_error_rate'],
            'history': history
        }, f'{OUTPUT_DIR}/best_model.pth')
        print(f"\n✅ Best model saved (Val Acc: {best_val_acc:.4f})")

    # Save best model (by 6→0 error rate)
    if val_metrics['six_to_zero_error_rate'] < best_six_to_zero_rate:
        best_six_to_zero_rate = val_metrics['six_to_zero_error_rate']
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_acc': val_metrics['accuracy'],
            'val_loss': val_metrics['loss'],
            'six_to_zero_error_rate': val_metrics['six_to_zero_error_rate'],
            'history': history
        }, f'{OUTPUT_DIR}/best_six_to_zero_model.pth')
        print(f"✅ Best 6→0 model saved (Error rate: {best_six_to_zero_rate:.2%})")

print("\n" + "="*70)
print("🎉 TRAINING COMPLETE!")
print("="*70)
print(f"\n📈 Final Results:")
print(f"   Best Val Accuracy: {best_val_acc:.2%}")
print(f"   Best 6→0 Error Rate: {best_six_to_zero_rate:.2%}")
print(f"\n📁 Models saved to: {OUTPUT_DIR}")
print(f"   - best_model.pth: Best overall accuracy")
print(f"   - best_six_to_zero_model.pth: Best 6→0 error rate")

## 📊 Visualize Training History

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot training & validation loss
axes[0, 0].plot(history['train_loss'], label='Train Loss')
axes[0, 0].plot(history['val_loss'], label='Val Loss')
axes[0, 0].set_title('Training & Validation Loss')
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Loss')
axes[0, 0].legend()
axes[0, 0].grid(True)

# Plot validation accuracy
axes[0, 1].plot(history['val_acc'], label='Val Accuracy', color='green')
axes[0, 1].set_title('Validation Accuracy')
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('Accuracy')
axes[0, 1].legend()
axes[0, 1].grid(True)

# Plot 6→0 error rate
axes[1, 0].plot(history['six_to_zero'], label='6→0 Error Rate', color='red')
axes[1, 0].set_title('6→0 Error Rate (Target: <5%)')
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylabel('Error Rate')
axes[1, 0].axhline(y=0.05, color='orange', linestyle='--', label='Target (5%)')
axes[1, 0].legend()
axes[1, 0].grid(True)

# Summary
axes[1, 1].axis('off')
summary_text = f"""
📊 Training Summary
==================

Best Val Accuracy: {best_val_acc:.2%}
Best 6→0 Error Rate: {best_six_to_zero_rate:.2%}

🎯 Targets:
- Accuracy: >90%
- 6→0 Error: <5%

✅ Status:
"""
if best_val_acc >= 0.90:
    summary_text += "\n✅ Accuracy target achieved!"
else:
    summary_text += f"\n⚠️ Accuracy: {(best_val_acc/0.90)*100:.1f}% of target"

if best_six_to_zero_rate <= 0.05:
    summary_text += "\n✅ 6→0 Error target achieved!"
else:
    summary_text += f"\n⚠️ 6→0 Error: {(best_six_to_zero_rate/0.05)*100:.1f}% of target"

axes[1, 1].text(0.1, 0.5, summary_text, fontsize=12, family='monospace',
              verticalalignment='center')

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/training_history.png', dpi=150)
plt.show()

print(f"✅ Training history plot saved to {OUTPUT_DIR}/training_history.png")

## 📤 Copy Trained Models Back to Drive

In [ ]:
# Copy models back to Google Drive
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/WaterMeter_Project/Project/model/M4_trained_v2"
!mkdir -p {DRIVE_OUTPUT_DIR}

print("="*70)
print("COPYING TRAINED MODELS TO GOOGLE DRIVE")
print("="*70)
print()
print("Copying trained models to Google Drive...")
!cp {OUTPUT_DIR}/best_model.pth {DRIVE_OUTPUT_DIR}/
!cp {OUTPUT_DIR}/best_six_to_zero_model.pth {DRIVE_OUTPUT_DIR}/
!cp {OUTPUT_DIR}/training_history.png {DRIVE_OUTPUT_DIR}/

print(f"\n✅ All files saved to: {DRIVE_OUTPUT_DIR}")
print("\n📋 Trained from scratch - Model files:")
print("  - best_model.pth: Best overall accuracy")
print("  - best_six_to_zero_model.pth: Best 6→0 error rate")
print("  - training_history.png: Training curves")
print("\n🚀 You can now use these models in your pipeline!")
print("\n💡 Usage in pipeline:")
print(f"   M4_MODEL = '{DRIVE_OUTPUT_DIR}/best_six_to_zero_model.pth'")
print()
print("="*70)
print("TRAINING FROM SCRATCH COMPLETE!")
print("="*70)

## 🎯 Summary - Training from Scratch with 3-Layer Strategy

### ⚠️ Training Type: FROM SCRATCH (No Pre-trained Weights)

This notebook trains a **brand new model from scratch** using:
- Random initialization (no transfer learning)
- MobileNet V1 + MultiHead (CTC + SAR) architecture
- 3-Layer Strategy for fixing 6→0 errors
- Higher learning rate (1e-3) for scratch training
- ~30-45 minutes training time on GPU T4

### ✅ 3-Layer Strategy Applied:

**Layer 1: WeightedRandomSampler** 🍽️
- Balanced "diet" - AI now sees rare digits (6,7,8,9) equally with common digits
- Can no longer "guess" 0 for easy probability

**Layer 2: Enhanced Sharpening** 🔍
- Applied Unsharp Masking + CLAHE + Denoising
- Made the "hook" on top of 6 more visible
- Improved local contrast for all digits

**Layer 3: Weighted Loss** ⚖️
- 📊 **Optimized weights based on actual failure analysis:**
  - Digit 6: **4.0x** (8x higher than digit 0) - Target chính!
  - Digit 1: **3.5x** (370 errors - cao nhất về số lượng)
  - Digit 8: **3.0x** (nhầm 0 và 1 nhiều)
  - Digit 9: **2.5x** (ít dữ liệu,容易被 bỏ qua)
  - Digits 4,5,7: **2.0x** (hay nhầm chéo)
  - Digits 2,3: **1.5x** (ổn định nhưng cần cải thiện)
  - Digit 0: **0.5x** (giảm trọng - AI ngừng đoán mò)

### 🏗️ Architecture: MobileNet V1 + MultiHead (CTC + SAR)

**MobileNet V1 Enhanced Backbone:**
- Depthwise separable convolutions (efficient)
- Multi-scale feature extraction
- Lightweight but powerful

**MultiHead (CTC + SAR):**
- CTC Head: For sequence alignment
- SAR Head: Attention-based decoding (better for 6→0 fix)
- Ensemble decoding: Combines both for best results

**Enhanced Input Processing:**
- RGB input [3, 48, 192] instead of grayscale
- Wider input for better digit separation
- Denoising + Sharpening + CLAHE

### 📈 Expected Results (Training from Scratch):
- **6→0 Error Rate**: 20.4% → **<5%**
- **1→0 Error Rate**: 19.2% → **<5%**
- **8→0 Error Rate**: 9.7% → **<3%**
- **Overall OCR Accuracy**: 76.55% → **>90%**
- **Pipeline Success**: 96.25% → **>98%**

### 🚀 Next Steps:
1. ✅ Copy trained models to local machine
2. Update pipeline script to use new model
3. Run full pipeline to verify improvements
4. Analyze new failure patterns (if any)
5. Deploy to production

---

**Training Time**: ~30-45 minutes on GPU T4
**Cost**: Free on Colab 💰

### 📊 Why Training from Scratch Works Better:

1. **Fresh Learning** - No bias from old model's mistakes
2. **Architecture Mismatch** - Old model (CRNN) vs New model (MobileNet + MultiHead)
3. **Higher LR** - 1e-3 allows faster learning from scratch
4. **Better Feature Extraction** - MobileNet V1 extracts features differently than old CNN
5. **Multi-Head Attention** - SAR head learns from scratch to focus on digit 6's "hook"

### 🎓 Key Insights

> "Training from scratch allows the model to learn the 3-Layer Strategy from day one.
>  The weighted loss, balanced sampling, and sharpening work together to teach the
>  model to recognize digit 6 correctly from the beginning, rather than trying to
>  unlearn old habits from a pre-trained model."

> "The SAR attention mechanism specifically targets the 6→0 error by focusing on
>  the 'hook' feature at the top of digit 6. When trained from scratch, it learns
>  this feature from the ground up, rather than trying to adapt existing features."

### 📝 Trained Model Files:

- `best_model.pth` - Best overall validation accuracy
- `best_six_to_zero_model.pth` - Best 6→0 error rate (recommended for production)
- `training_history.png` - Training curves visualization

### 💡 Differences from Fine-tuning:

| Aspect | Fine-tuning | Training from Scratch |
|--------|-------------|----------------------|
| **Initial Weights** | Pre-trained (old model) | Random initialization |
| **Learning Rate** | 1e-5 (very low) | 1e-3 (higher) |
| **Training Time** | ~20 minutes | ~30-45 minutes |
| **Architecture** | Same as old model | New (MobileNet + MultiHead) |
| **Unlearning Old Habits** | Difficult | N/A (fresh start) |
| **Final Accuracy** | Good (~82%) | Better (>90% expected) |

### 🎯 Recommendation:

**Use Training from Scratch when:**
- Architecture changes significantly
- Old model has strong biases (like 6→0 errors)
- You have sufficient training data
- You have time for longer training (30-45 min)

**Use Fine-tuning when:**
- Architecture remains the same
- Old model performs well but needs minor tweaks
- Limited training time available